# We want to find the "closest" Pauli channel to the error channel

https://arxiv.org/pdf/2311.09129.pdf

In [52]:
import qutip
import numpy as np
import pickle 
import os
from utils import compute_and_store_2_level_dm,generate_single_mapping
import scqubits

zero = qutip.basis(2, 0)
one = qutip.basis(2, 1)
plus= (zero +  one).unit()
minus = (zero - one).unit()
states_ideal  = [zero,
            one,
            plus,
            minus]


def get_phase_corrected_two_level_final_dms(filename):
    ntraj = 400

    EJ = 3
    EC = 0.6
    EL = 0.13
    Er = 7.2622522
    g_strength = 0.3
    qubit_level = 30
    osc_level = 30

    qbt = scqubits.Fluxonium(EJ=EJ,EC=EC,EL=EL,flux=0,cutoff=110,truncated_dim=qubit_level)
    osc = scqubits.Oscillator(E_osc=Er,truncated_dim=osc_level)
    hilbertspace = scqubits.HilbertSpace([qbt, osc])
    hilbertspace.add_interaction(g_strength=g_strength,op1=qbt.n_operator,op2=osc.creation_operator,add_hc=True)
    hilbertspace.generate_lookup()
    product_to_dressed = generate_single_mapping(hilbertspace.hamiltonian())

    products_to_keep = []

    for ql in [0]:
        for ol in range(30):
            products_to_keep.append([ql,ol])
    for ql in range(1,3):
        for ol in range(10):
            products_to_keep.append([ql,ol])

    with open(filename, 'rb') as file:
        tomo_results = pickle.load(file)

    dm_tomo_results_list = []
    for result in tomo_results:
        new_result = qutip.solver.Result()
        new_result.times = [result.times[-1]]
        states_array = np.array([[traj[-1].full()] for traj in result.states])
        summed_dm_array = np.einsum('ntrc,ntij->tri', states_array, states_array.conj()) # n is traj index, t is time index, r is row index, c is column index, i and j are the row and column index of the conjugated ket
        averaged_dm_array = summed_dm_array/ntraj
        new_result.states = [qutip.Qobj(dm) for dm in averaged_dm_array]
        dm_tomo_results_list.append(new_result)

    import shutil
    dir_name = 'temp_tomo'
    if os.path.exists(dir_name):
        shutil.rmtree(dir_name)
    os.mkdir(dir_name)
    tasks = [(dm_tomo_results_list, 
            f'{dir_name}/res_{i}_state_{j}.pkl',
            i, j,  product_to_dressed, qubit_level, osc_level, 1, 2,products_to_keep) 
            for i in range(len(dm_tomo_results_list)) 
            for j in range(len(dm_tomo_results_list[i].states))]

    from multiprocessing import Pool

    with Pool(processes=6) as pool:
        pool.map(compute_and_store_2_level_dm, tasks)

    num_initial_states = len(dm_tomo_results_list)
    num_time_steps = len(dm_tomo_results_list[0].times)
    two_level_states = []
    for i in range(num_initial_states) :
        two_level_states.append([])
        for j in range(num_time_steps):
            with open(f'{dir_name}/res_{i}_state_{j}.pkl', 'rb') as f:
                state = pickle.load(f)
            two_level_states[-1].append(state.unit())


    def calc_average_fidelity_with_phase(phase,dms,states_ideal):
        gate = qutip.qip.operations.phasegate(phase)
        fid=[]
        for dm,state_ideal in zip(dms,states_ideal):
            # new_dm = gate*dm*gate.dag()
            fid.append(qutip.fidelity(dm, gate*state_ideal))
        return 1-sum(fid)/len(fid)


    from scipy.optimize import minimize


    dms = [states[-1] for states in two_level_states]
    def objective_function(phase):
        return calc_average_fidelity_with_phase(phase[0], dms, states_ideal)
    initial_phase = [0.0]
    bounds = [(0, 2 * 3.141592653589793)]
    opt_result = minimize(objective_function, initial_phase,method="COBYLA")
    infidelity = opt_result.fun
    phase = opt_result.x[0]

    print(f'loaded {filename},corrective phase:{phase},infidelity{infidelity}')
    gate = qutip.qip.operations.phasegate(-phase)
    return [gate*dm*gate.dag() for dm in dms]

In [53]:
dms_kappa1em3 = get_phase_corrected_two_level_final_dms('mcsolve_tomo_results_kappa1e-3.pkl')
dms_kappa2em3 = get_phase_corrected_two_level_final_dms('mcsolve_tomo_results_kappa2e-3.pkl')
dms_kappa5em3 = get_phase_corrected_two_level_final_dms('mcsolve_tomo_results_kappa5e-3.pkl')

loaded mcsolve_tomo_results_kappa1e-3.pkl,corrective phase:-2.1125046875000004,infidelity0.0002449447553671824
loaded mcsolve_tomo_results_kappa2e-3.pkl,corrective phase:-2.107321875,infidelity0.0007661243621651659
loaded mcsolve_tomo_results_kappa5e-3.pkl,corrective phase:-2.06015859375,infidelity0.007183499726782228


In [58]:
dms_kappa1em3[0]

Quantum object: dims = [[2], [2]], shape = (2, 2), type = oper, isherm = True
Qobj data =
[[ 9.99999909e-01+0.00000000e+00j -1.34803929e-05+5.30374797e-06j]
 [-1.34803929e-05-5.30374797e-06j  9.10762724e-08+0.00000000e+00j]]

In [59]:
dms_kappa1em3[1]

Quantum object: dims = [[2], [2]], shape = (2, 2), type = oper, isherm = True
Qobj data =
[[1.03013073e-07+0.00000000e+00j 2.60596492e-06+8.90690317e-06j]
 [2.60596492e-06-8.90690317e-06j 9.99999897e-01+0.00000000e+00j]]

In [60]:
dms_kappa1em3[2]

Quantum object: dims = [[2], [2]], shape = (2, 2), type = oper, isherm = True
Qobj data =
[[0.49971892+0.00000000e+00j 0.499026  -6.24226695e-05j]
 [0.499026  +6.24226695e-05j 0.50028108+0.00000000e+00j]]

In [61]:
dms_kappa1em3[3]

Quantum object: dims = [[2], [2]], shape = (2, 2), type = oper, isherm = True
Qobj data =
[[ 0.50000861+0.00000000e+00j -0.4990151 +4.96751129e-05j]
 [-0.4990151 -4.96751129e-05j  0.49999139+0.00000000e+00j]]

## Michael's method require operating on a wierd density matrix as the initial state, which I can't do with mcsolve,

## He does have a work around, which is just approximate the channel as identity and a discrete decay channel but how should we express this decay channel?

In [36]:
initial_states = []
def get_pauli_model(dms):
    def get_result_dm(reference_state):
        if reference_state == qutip.ket2dm(zero):
            return dms[0]
        elif reference_state == qutip.ket2dm(one):
            return dms[1]
        elif reference_state == qutip.ket2dm(plus):
            return dms[2]
        else: #reference_state == qutip.ket2dm(minus):
            return dms[3]
        
    def uPauli_for_noise_sampling(Pauli,U,U0,state1,state2):
        ################################################
        # This method is from an unpublished work of someone else
        ################################################
        temp = state1*state2.dag()
        temp = U0.inv() * temp * U0.inv().dag()
        initial_states.append(temp)
        mixed = U(temp)
        temp1 = Pauli * mixed * Pauli.dag()
        temp1 = state1.dag() * temp1  * state2

        return temp1[0,0]


    d_mixed = {'X':[],'Y':[],"Z":[]}
    for state1_name, state1 in zip(['0','1','+','-'],states_ideal):
        for state2_name, state2 in zip(['0','1','+','-'],states_ideal):
            for error_name,Pauli in zip(['X','Y','Z'],[qutip.sigmax(),qutip.sigmay(),qutip.sigmaz()]):
                uP_mixed = uPauli_for_noise_sampling(Pauli = Pauli,
                    U = get_result_dm,
                    U0 = qutip.qeye(2),
                    state1 = state1,
                    state2 = state2)
                d_mixed[error_name].append(uP_mixed)

    for key, val in d_mixed.items():
        print(key+' '+str(np.square(abs(np.mean(val)))))


In [62]:
get_pauli_model(dms_kappa1em3)